# Machine Learning - Credit Risk

Notebook inicial para classificacao e regressao com Scikit-Learn.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

load_dotenv(Path('..') / '.env')
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER', 'postgres')}:{os.getenv('DB_PASSWORD', '')}@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '5432')}/{os.getenv('DB_NAME', 'credit_risk')}"
)

query = '''
SELECT
    f.id_aplicacao,
    c.idade,
    c.renda_anual,
    c.posse_residencia,
    c.tempo_empregada,
    p.finalidade_emprestimo,
    p.classificacao_emprestimo,
    h.inadimplencia_historica,
    h.duracao_historico_credito,
    f.montante_emprestimo,
    f.juros_aplicado,
    f.porcentagem_renda,
    f.status_emprestimo
FROM analytics.fato_emprestimo f
JOIN analytics.dim_cliente c ON c.id_cliente = f.id_cliente
JOIN analytics.dim_perfil_emprestimo p ON p.id_perfil_emprestimo = f.id_perfil_emprestimo
JOIN analytics.dim_historico_credito h ON h.id_historico_credito = f.id_historico_credito
'''

df = pd.read_sql(query, engine)
df.head()

In [ ]:
target = 'status_emprestimo'
X = df.drop(columns=[target])
y = df[target]

categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(exclude='object').columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'knn': KNeighborsClassifier(n_neighbors=5),
    'decision_tree': DecisionTreeClassifier(random_state=42),
    'random_forest': RandomForestClassifier(random_state=42),
    'logistic_regression': LogisticRegression(max_iter=1000),
}

for name, model in models.items():
    clf = Pipeline([('preprocessor', preprocessor), ('model', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(name)
    print('accuracy:', accuracy_score(y_test, y_pred))
    print('precision:', precision_score(y_test, y_pred, zero_division=0))
    print('recall:', recall_score(y_test, y_pred, zero_division=0))
    print('f1:', f1_score(y_test, y_pred, zero_division=0))
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))
    print('-' * 40)

In [ ]:
regression_df = df.dropna(subset=['juros_aplicado']).copy()
X_reg = regression_df.drop(columns=['juros_aplicado'])
y_reg = regression_df['juros_aplicado']

X_reg = X_reg.drop(columns=['status_emprestimo'])
categorical_cols_reg = X_reg.select_dtypes(include='object').columns.tolist()
numeric_cols_reg = X_reg.select_dtypes(exclude='object').columns.tolist()

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols_reg),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols_reg),
    ]
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
reg_pipeline = Pipeline([('preprocessor', preprocessor_reg), ('model', LinearRegression())])
reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg = reg_pipeline.predict(X_test_reg)

rmse = mean_squared_error(y_test_reg, y_pred_reg, squared=False)
r2 = r2_score(y_test_reg, y_pred_reg)
print({'rmse': rmse, 'r2': r2})